In [2]:
import torch

In [8]:
'''
Lets build a wavenet from scractch.

Plan:
1. naive sliding-window dataset
2. manual 1D convolution
3. Conv1d version
4. causal convolution
5. dilated causal convolution
6. stacked dilated layers
7. residual + skip block
8. tiny WaveNet-like model

''';

In [7]:
'''
Given:

x = torch.tensor([2., 5., 1., 7., 3., 4.])
w = torch.tensor([0.1, 0.2, 0.3])

Let's write a function:
def conv1d_manual(x, w):
    ...

That should output:
0.1*2 + 0.2*5 + 0.3*1
0.1*5 + 0.2*1 + 0.3*7
0.1*1 + 0.2*7 + 0.3*3
0.1*7 + 0.2*3 + 0.3*4
''';

In [ ]:
x = torch.tensor([2., 5., 1., 7., 3., 4.])
w = torch.tensor([0.1, 0.2, 0.3])

def conv1d_manual(x, w):
    next_layer = []
    input_size = len(x)
    kernel_size = len(w)
    
    for i in range(input_size - (kernel_size - 1)):
        next_layer.append(torch.dot(w, x[i:i+kernel_size]))
    next_layer = torch.tensor(next_layer)
    
    return next_layer

print(conv1d_manual(x, w))

In [ ]:
'''
Let's add some more detectors
'''

In [36]:
x = torch.tensor([2., 5., 1., 7., 3., 4.])
W = torch.randn(32, 3) # 32 detectors with a kernel size of 3 each

In [40]:
def conv1d_manual(inputs, detectors):
    next_layer = []
    input_size = len(inputs)
    kernel_size = len(detectors[0])
    
    for detector in detectors:
        detector_layer = []
        for i in range(input_size - (kernel_size - 1)):
            detector_layer.append(torch.dot(detector, inputs[i:i+kernel_size]))
        detector_layer = torch.stack(detector_layer)
        next_layer.append(detector_layer)
    
    next_layer = torch.stack(next_layer)
    return next_layer

out = conv1d_manual(x, W)
print(out, out.shape)

tensor([[ -8.3370,  -5.8544, -13.8550,  -5.6411],
        [  0.2055,   1.5657,   6.7003,  -6.4654],
        [  3.1064,  11.7278,   5.4410,   9.2643],
        [ -9.9005,   8.2193,  -8.4885,  -4.4498],
        [ -4.6923,  14.4397,  -8.4301,  11.5270],
        [  1.5304,   5.8445,   0.6900,   7.0337],
        [ -1.0307,  -4.4923,  -2.1665,  -3.1076],
        [  5.4040,  -2.3677,   6.3133,   2.0536],
        [ -8.2196, -11.0730, -13.6004,  -9.7988],
        [  9.2357,  -4.7446,  10.6332,   3.1512],
        [  5.2357,   9.2543,   6.9131,  10.0963],
        [  1.8603,  -1.2625,   2.7389,  -0.3314],
        [  5.0073,  -3.5367,   4.4752,   2.5169],
        [ -8.2182, -13.0308, -14.9413,  -9.7060],
        [ -3.5240,   2.2310,   0.6655,  -6.6066],
        [ -2.5662,   1.8980,  -2.4346,  -1.0515],
        [  7.1862,  12.6397,  12.4833,  10.1720],
        [ -4.6949,  -0.6732,  -6.8309,  -2.2949],
        [ -5.7790,   4.6967,  -6.3854,  -0.9395],
        [-11.4665,  -5.1310, -16.9581,  -8.0105],


In [69]:
'''
But we can make this even more efficient by using vectorization. 
'''

W.shape, x.shape, x[0:3].shape

(torch.Size([32, 3]), torch.Size([6]), torch.Size([3]))

In [70]:
def conv1d_manual(inputs, detectors):
    next_layer = []
    input_size = len(inputs)
    kernel_size = detectors.shape[1]
    out_size = input_size - (kernel_size - 1)

    for i in range(out_size):
        window = inputs[i:i + kernel_size]
        out = detectors @ window
        next_layer.append(out)

    next_layer = torch.stack(next_layer)
    
    return next_layer.T

out = conv1d_manual(x, W)
print(out, out.shape)

tensor([[ -8.3370,  -5.8544, -13.8550,  -5.6411],
        [  0.2055,   1.5657,   6.7003,  -6.4654],
        [  3.1064,  11.7278,   5.4410,   9.2643],
        [ -9.9005,   8.2193,  -8.4885,  -4.4498],
        [ -4.6923,  14.4397,  -8.4301,  11.5270],
        [  1.5304,   5.8445,   0.6900,   7.0337],
        [ -1.0307,  -4.4923,  -2.1665,  -3.1076],
        [  5.4040,  -2.3677,   6.3133,   2.0536],
        [ -8.2196, -11.0730, -13.6004,  -9.7988],
        [  9.2357,  -4.7446,  10.6332,   3.1512],
        [  5.2357,   9.2543,   6.9131,  10.0963],
        [  1.8603,  -1.2625,   2.7389,  -0.3314],
        [  5.0073,  -3.5367,   4.4752,   2.5169],
        [ -8.2182, -13.0308, -14.9413,  -9.7060],
        [ -3.5240,   2.2310,   0.6655,  -6.6066],
        [ -2.5662,   1.8980,  -2.4346,  -1.0515],
        [  7.1862,  12.6397,  12.4833,  10.1720],
        [ -4.6949,  -0.6732,  -6.8309,  -2.2949],
        [ -5.7790,   4.6967,  -6.3854,  -0.9395],
        [-11.4665,  -5.1310, -16.9581,  -8.0105],


In [ ]:
'''
Notice we transposed at the end. When we stacked the 32 features over the 4 time steps,
we were given a 4, 32 tensor. It tell us that at each time, what were the activations
of each feature detector?

So we transposed it, and we got a tensor of 32, 4. Now this is telling us: for each feature,
how active was it over time (over each time step)?

Notice that at the beginning, we had a 1D tensor of inputs.
But now, we have a 2D tensor of local feature activations over time to pass to our next layer.
''